# Order-Spectrum Walkthrough (Healthy vs Looseness)

This notebook demonstrates the **exact preprocessing pipeline implemented in the framework** (under `src/`) step by step:

1. Orientation standardization (axis reorder + optional sign flips)
2. Resampling to a fixed sampling rate
3. Windowing
4. RPM selection (metadata vs estimator)
5. Order-spectrum projection (FFT → order axis → interpolation)
6. Optional global z-normalization

The code **consumes functions/classes from the framework**; it does not reimplement preprocessing.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

# Ensure repo root is importable
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.training.config import load_config
from src.data.dataloader import Part3Metadata, read_signal_csv, standardize_orientation
from src.preprocessing.preprocessor import Preprocessor, resample_to_frequency

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True

In [ ]:
# Parameters
CONFIG_PATH = REPO_ROOT / 'configs' / 'order_spectrum_conv_net_rpm.yaml'
# Optionally override these if you know the exact label strings in metadata
LABEL_A = None
LABEL_B = None
NORMALIZER_FIT_SAMPLES = 200

cfg = load_config(CONFIG_PATH)
cfg.train_metadata_path, cfg.train_data_dir, cfg.output_order

In [ ]:
# Load metadata and inspect label distribution
meta = Part3Metadata(cfg.train_metadata_path)
samples_all = list(meta.iter_samples())
labeled = [s for s in samples_all if s.condition is not None]

label_counts = Counter([str(s.condition) for s in labeled])
label_counts

In [ ]:
# Choose two labels
labels_sorted = [k for k, _ in label_counts.most_common()]
if LABEL_A is None:
    LABEL_A = labels_sorted[0] if len(labels_sorted) > 0 else None
if LABEL_B is None:
    LABEL_B = labels_sorted[1] if len(labels_sorted) > 1 else None

LABEL_A, LABEL_B

In [ ]:
# Pick one on-disk sample from each label
def pick_first_sample(label: str):
    for s in labeled:
        if str(s.condition) == str(label):
            csv_path = cfg.train_data_dir / f'{s.sample_id}.csv'
            if csv_path.exists():
                return s
    raise FileNotFoundError(f'Could not find any on-disk sample for label={label!r}')

sA = pick_first_sample(LABEL_A)
sB = pick_first_sample(LABEL_B)

(sA.sample_id, sA.condition, sA.rpm), (sB.sample_id, sB.condition, sB.rpm)

In [ ]:
# Build preprocessors from config
pre_cfg = cfg.preprocessing

def make_preprocessor(*, feature_mode: str, z_norm: bool) -> Preprocessor:
    return Preprocessor(
        downsample_hz=float(pre_cfg.get('downsample_hz', 100.0)),
        window_seconds=float(pre_cfg.get('window_seconds', 0.05)),
        step_seconds=pre_cfg.get('step_seconds', None),
        z_norm=bool(z_norm),
        feature_mode=str(feature_mode),
        rpm_policy=str(pre_cfg.get('rpm_policy', 'auto')),
        rpm_min=float(pre_cfg.get('rpm_min', 300.0)),
        rpm_max=float(pre_cfg.get('rpm_max', 6000.0)),
        rpm_discrepancy_tol=float(pre_cfg.get('rpm_discrepancy_tol', 0.2)),
        rpm_top_k=int(pre_cfg.get('rpm_top_k', 8)),
        rpm_harmonics=int(pre_cfg.get('rpm_harmonics', 5)),
        order_max=float(pre_cfg.get('order_max', 20.0)),
        order_bins=int(pre_cfg.get('order_bins', 128)),
        order_log_power=bool(pre_cfg.get('order_log_power', True)),
        order_per_window_standardize=bool(pre_cfg.get('order_per_window_standardize', True)),
    )

pre_time = make_preprocessor(feature_mode='time', z_norm=False)
pre_order_raw = make_preprocessor(feature_mode='order_spectrum', z_norm=False)
pre_order_norm = make_preprocessor(feature_mode='order_spectrum', z_norm=True)

def iter_fit_samples():
    for s in labeled[:int(NORMALIZER_FIT_SAMPLES)]:
        t, sig = read_signal_csv(cfg.train_data_dir / f'{s.sample_id}.csv')
        sig = standardize_orientation(sig, s.orientation, output_order=cfg.output_order)
        yield t, sig, s.rpm

pre_order_norm.fit(times_and_signals=iter_fit_samples())

pre_time, pre_order_raw, pre_order_norm

## Walkthrough on two samples
We run the same steps for one sample from each label.

In [ ]:
def show_sample_walkthrough(s):
    sample_id = str(s.sample_id)
    label = None if s.condition is None else str(s.condition)
    rpm_meta = None if s.rpm is None else float(s.rpm)

    t, sig_raw = read_signal_csv(cfg.train_data_dir / f'{sample_id}.csv')
    sig_std = standardize_orientation(sig_raw, s.orientation, output_order=cfg.output_order)

    # 1) Raw standardized time signal
    fig, ax = plt.subplots(1, 1)
    for ci, ch in enumerate(cfg.output_order):
        ax.plot(t, sig_std[:, ci], label=ch)
    ax.set_title(f'Raw (standardized orientation) — sample_id={sample_id}, label={label}')
    ax.set_xlabel('time')
    ax.set_ylabel('acc')
    ax.legend()
    plt.show()

    # 2) Resampled signal
    t_ds, sig_ds = resample_to_frequency(np.asarray(t), np.asarray(sig_std), target_hz=float(pre_cfg.get('downsample_hz', 100.0)))
    fig, ax = plt.subplots(1, 1)
    for ci, ch in enumerate(cfg.output_order):
        ax.plot(t_ds, sig_ds[:, ci], label=ch)
    ax.set_title('Resampled signal')
    ax.set_xlabel('time')
    ax.set_ylabel('acc (resampled)')
    ax.legend()
    plt.show()

    # 3) RPM metadata vs estimator
    rpm_est = pre_order_raw.estimate_rpm(time=np.asarray(t), signals=np.asarray(sig_std))
    print({'rpm_metadata': rpm_meta, 'rpm_estimated': rpm_est, 'rpm_policy': pre_cfg.get('rpm_policy', 'auto')})

    # 4) Time windows (for visualization)
    w_time, rpm_used_time = pre_time.transform_with_rpm(time=np.asarray(t), signals=np.asarray(sig_std), rpm=rpm_meta)
    print('time windows:', w_time.shape, 'rpm_used:', rpm_used_time)
    if w_time.shape[0] > 0:
        w0 = w_time[0]
        x = np.arange(w0.shape[0]) / float(pre_cfg.get('downsample_hz', 100.0))
        fig, ax = plt.subplots(1, 1)
        for ci, ch in enumerate(cfg.output_order):
            ax.plot(x, w0[:, ci], label=ch)
        ax.set_title('First time-domain window (before projection)')
        ax.set_xlabel('seconds within window')
        ax.set_ylabel('acc')
        ax.legend()
        plt.show()

    # 5) Order-spectrum windows
    w_order_raw, rpm_used = pre_order_raw.transform_with_rpm(time=np.asarray(t), signals=np.asarray(sig_std), rpm=rpm_meta)
    print('order-spectrum windows:', w_order_raw.shape, 'rpm_used:', rpm_used)

    order_max = float(pre_cfg.get('order_max', 10.0))
    order_bins = int(pre_cfg.get('order_bins', 128))
    order_axis = np.linspace(0.0, order_max, order_bins)

    if w_order_raw.shape[0] > 0:
        o0 = w_order_raw[0]
        fig, ax = plt.subplots(1, 1)
        for ci, ch in enumerate(cfg.output_order):
            ax.plot(order_axis, o0[:, ci], label=ch)
        ax.set_title('First projected window (order-spectrum features)')
        ax.set_xlabel('Order (× shaft speed)')
        ax.set_ylabel('feat (order)')
        ax.legend()
        plt.show()

        mean_o = w_order_raw.mean(axis=0)
        fig, ax = plt.subplots(1, 1)
        for ci, ch in enumerate(cfg.output_order):
            ax.plot(order_axis, mean_o[:, ci], label=ch)
        ax.set_title('Mean order-spectrum feature across windows')
        ax.set_xlabel('Order (× shaft speed)')
        ax.set_ylabel('feat (order)')
        ax.legend()
        plt.show()

    # 6) After global z_norm
    w_order_norm, _ = pre_order_norm.transform_with_rpm(time=np.asarray(t), signals=np.asarray(sig_std), rpm=rpm_meta)
    if w_order_norm.shape[0] > 0:
        mean_n = w_order_norm.mean(axis=0)
        fig, ax = plt.subplots(1, 1)
        for ci, ch in enumerate(cfg.output_order):
            ax.plot(order_axis, mean_n[:, ci], label=ch)
        ax.set_title('Mean order-spectrum after global z_norm')
        ax.set_xlabel('Order (× shaft speed)')
        ax.set_ylabel('z-norm feat')
        ax.legend()
        plt.show()

    return {
        'sample_id': sample_id,
        'label': label,
        'rpm_metadata': rpm_meta,
        'rpm_estimated': rpm_est,
        'rpm_used': rpm_used,
        'n_windows': int(w_order_raw.shape[0]),
        'mean_order_feat': None if w_order_raw.shape[0] == 0 else w_order_raw.mean(axis=0),
    }

summary_A = show_sample_walkthrough(sA)
summary_B = show_sample_walkthrough(sB)
summary_A, summary_B

## Overlay comparison
We overlay the **mean order-spectrum feature** for the two selected samples.

In [ ]:
order_max = float(pre_cfg.get('order_max', 10.0))
order_bins = int(pre_cfg.get('order_bins', 128))
order_axis = np.linspace(0.0, order_max, order_bins)

mA = summary_A['mean_order_feat']
mB = summary_B['mean_order_feat']

if mA is None or mB is None:
    print('One of the samples produced zero windows; adjust window_seconds/step_seconds.')
else:
    for ci, ch in enumerate(cfg.output_order):
        fig, ax = plt.subplots(1, 1)
        ax.plot(order_axis, mA[:, ci], label=f"{summary_A['label']} ({summary_A['sample_id']})")
        ax.plot(order_axis, mB[:, ci], label=f"{summary_B['label']} ({summary_B['sample_id']})")
        ax.set_title(f'Mean order-spectrum — channel={ch}')
        ax.set_xlabel('Order (× shaft speed)')
        ax.set_ylabel('feat (order)')
        ax.legend()
        plt.show()